In [23]:
import os
import pandas as pd
from hashlib import md5
from PIL import Image

In [24]:
data_path = "/srv/defectDetectionDataset/surfaceClassification/full_clean_other"
artifacts_path = "./artifacts/clean"

In [25]:
train_images = []

extensions = [".png", ".jpg", ".jpeg"]
train_path = os.path.join(data_path, "train")

for dirpath, dirnames, filenames in os.walk(train_path):
    for filename in filenames:
        if any(filename.endswith(ext) for ext in extensions):
            file_path = os.path.join(dirpath, filename)
            label = os.path.basename(dirpath)
            img = Image.open(file_path)
            file_path = file_path.removeprefix(data_path)
            image_id = md5(img.tobytes()).hexdigest()
            train_images.append({"image_id": image_id, "file_path": file_path, "label": label})
            
all_df = pd.DataFrame(train_images)

In [26]:
val_images = []
val_path = os.path.join(data_path, "val")

for dirpath, dirnames, filenames in os.walk(val_path):
    for filename in filenames:
        if any(filename.endswith(ext) for ext in extensions):
            file_path = os.path.join(dirpath, filename)
            img = Image.open(file_path)
            image_id = md5(img.tobytes()).hexdigest()
            val_images.append({"image_id": image_id})

val_df = pd.DataFrame(val_images)

In [27]:
test_images = []
test_path = os.path.join(data_path, "test")

for dirpath, dirnames, filenames in os.walk(test_path):
    for filename in filenames:
        if any(filename.endswith(ext) for ext in extensions):
            file_path = os.path.join(dirpath, filename)
            img = Image.open(file_path)
            image_id = md5(img.tobytes()).hexdigest()
            test_images.append({"image_id": image_id})


test_df = pd.DataFrame(test_images)

In [28]:
print("All: ", len(all_df))
print("Val: ", len(val_df))
print("Test: ", len(test_df))

All:  4249
Val:  970
Test:  1176


In [29]:
# remove val and test images from train_df

train_df = all_df[~all_df["image_id"].isin(val_df["image_id"])]
train_df = train_df[~train_df["image_id"].isin(test_df["image_id"])]

total = len(train_df) + len(val_df) + len(test_df)

print(f"Train: {len(train_df)}")
print(f"Val: {len(val_df)}")
print(f"Test: {len(test_df)}")

print(f"Total: {total}")

Train: 4249
Val: 970
Test: 1176
Total: 6395


In [30]:
train_duplicates = train_df[train_df.duplicated(subset=["image_id"], keep=False)]
print(f"Train duplicates: {len(train_duplicates)}")

Train duplicates: 0


In [31]:
val_duplicates = val_df[val_df.duplicated(subset=["image_id"], keep=False)]
print(f"Val duplicates: {len(val_duplicates)}")

Val duplicates: 0


In [32]:
test_duplicates = test_df[test_df.duplicated(subset=["image_id"], keep=False)]
print(f"Test duplicates: {len(test_duplicates)}")

Test duplicates: 0


In [33]:
# Remove all images from the train directory, that are not in the train_df

import shutil

removed_path = os.path.join(data_path, "removed")
os.makedirs(removed_path, exist_ok=True)

moved_images = 0

for dirpath, dirnames, filenames in os.walk(train_path):
    for filename in filenames:
        if any(filename.endswith(ext) for ext in extensions):
            file_path = os.path.join(dirpath, filename)
            img = Image.open(file_path)
            image_id = md5(img.tobytes()).hexdigest()
            img.close()
            if image_id not in train_df["image_id"].values:
                shutil.move(file_path, os.path.join(removed_path, filename))
                moved_images += 1
                

In [34]:
print("Moved images: ", moved_images)

Moved images:  0


In [35]:
## images that were in noisy, but not in the full clean dataset

noisy_path = "/srv/defectDetectionDataset/surfaceClassification/noisy_clustered_new"

noisy_images = []

for dirpath, dirnames, filenames in os.walk(noisy_path):
    for filename in filenames:
        if any(filename.endswith(ext) for ext in extensions):
            file_path = os.path.join(dirpath, filename)
            label = os.path.basename(dirpath)
            img = Image.open(file_path)
            file_path = file_path.removeprefix(noisy_path)
            image_id = md5(img.tobytes()).hexdigest()
            noisy_images.append({"image_id": image_id, "file_path": file_path, "label": label})

noisy_df = pd.DataFrame(noisy_images)
print("Noisy images: ", len(noisy_df))

Noisy images:  6396


In [36]:
all_clean = pd.concat([train_df, val_df, test_df], ignore_index=True)
print("All clean images: ", len(all_clean))

All clean images:  6395


In [37]:
removed_imgs = noisy_df[~noisy_df["image_id"].isin(all_clean["image_id"])]
print("Removed images: ", len(removed_imgs))

Removed images:  1


## final overlap check

In [38]:
val_in_train = val_df[val_df["image_id"].isin(train_df["image_id"])]
print(f"Val images also in train: {len(val_in_train)}")

Val images also in train: 0


In [40]:
test_in_train = test_df[test_df["image_id"].isin(train_df["image_id"])]
print(f"Test images also in train: {len(test_in_train)}")

Test images also in train: 0
